# Annotation Agreement Metrics
This notebook shows how to compute various annotation agreement metrics using Python. We will cover Cohen's Kappa, Fleiss Kappa, Weighted Kappa, Intraclass Correlation Coefficient (ICC), Pearson Correlation, and Krippendorff's Alpha. Which metric to use depends on the number of annotators, the type of data (categorical, ordinal, continuous), and whether there are missing annotations.

Type of data:
- Categorical: nominal data with no inherent order (e.g. polite vs. impolite or spam vs. not spam)
- Ordinal: data with a specific order (e.g. likert scale from 1 to 5, e.g. for politeness level from 1 = polite to 5 = impolite)
- Continuous: data on a continuous scale (e.g. a score from 0 to 100 indicating level of politeness)


### Cohen's Kappa
If you have two annotators and they annotate the same set of items with a specific categorical label (e.g. polite vs. impolite), you can use cohen's kappa. A score of 1 means perfect agreement, a score of 0 means no agreement beyond chance, and negative values indicate disagreement.

In [ ]:
import pandas as pd
import random
# create a dataframe with random annotations from two annotators:
annotations_1 = [random.choice(['polite', 'impolite']) for _ in range(30)]
annotations_2 = [random.choice(['polite', 'impolite']) for _ in range(30)]
df_two_annotators_categorical = pd.DataFrame({'annotator_1': annotations_1, 'annotator_2': annotations_2})

In [ ]:
from sklearn.metrics import cohen_kappa_score
kappa = cohen_kappa_score(df_two_annotators_categorical['annotator_1'], df_two_annotators_categorical['annotator_2'])
print(f"Cohen's Kappa: {kappa}")

### Fleiss Kappa
If you have more than two annotators and still categorical annotations you can use Fleiss kappa. Similar to cohen's kappa, a score of 1 means perfect agreement, a score of 0 means no agreement beyond chance, and negative values indicate disagreement.

In [ ]:
annotations_3 = [random.choice(['polite', 'impolite']) for _ in range(30)]
df_three_annotators_categorical = pd.DataFrame({'annotator_1': annotations_1, 'annotator_2': annotations_2, 'annotator_3': annotations_3})

from statsmodels.stats.inter_rater import fleiss_kappa
# convert the dataframe to the required format. what is this format?
# we need to create a matrix where each row represents an item and each column represents the count of each label, e.g. 2 for the column 'polite' and 1 for the column 'impolite' if 2 annotators labeled the item as polite and 1 as impolite.
label_counts = []
for index, row in df_three_annotators_categorical.iterrows():
    polite_count = sum([1 for label in row if label == 'polite'])
    impolite_count = sum([1 for label in row if label == 'impolite'])
    label_counts.append([polite_count, impolite_count])
label_counts_df = pd.DataFrame(label_counts, columns=['polite', 'impolite'])


In [ ]:
label_counts_df.head()

In [ ]:
print("Fleiss Kappa:", fleiss_kappa(label_counts_df.values))

### Weighted Kappa

If we annotate items using a likert-scale, for example three levels of politeness (1 = polite, 2 = neutral , 3 = impolite) we can use weighted kappa to measure agreement between two annotators. This takes into account the degree of disagreement. For example if annotator 1 labels an item as 1 (polite) and annotator 2 labels it as 3 (impolite), this is a larger disagreement than if annotator 2 labeled it as 2 (neutral). The degree of disagreement can be weighted, either linearly ("weights='linear'") or quadratically ("weights='quadratic'").

In [ ]:
annotations_1 = [random.choice([1, 2, 3]) for _ in range(30)]
annotations_2 = [random.choice([1, 2, 3]) for _ in range(30)]
df_two_annotators_likert = pd.DataFrame({'annotator_1': annotations_1, 'annotator_2': annotations_2})


In [ ]:
weighted_kappa = cohen_kappa_score(df_two_annotators_likert['annotator_1'], df_two_annotators_likert['annotator_2'], weights='linear')
print(f"Weighted Kappa: {weighted_kappa}")

### Intraclass Correlation Coefficient (ICC) and Pearson Correlation

If we have annotations on a continuous scale, for example a score from 0 to 100 indicating the level of politeness, we can use the intraclass correlation coefficient (ICC) or the pearson correlation coefficient to measure agreement between annotators. A positive pearson correlation indicates that as one annotator's scores increase, the other annotator's scores also tend to increase. You can also check if the correlation is statistically significant by looking at the p-value.
For the ICC, a value close to 1 indicates high agreement, while a value close to 0 indicates low agreement. We have different types of ICC depending on the study design and the type of consistency we want to measure. Here we will use a two-way mixed effects model for absolute agreement (ICC(3,1)).


In [ ]:
annotations_1 = [random.uniform(0, 100) for _ in range(30)]
annotations_2 = [random.uniform(0, 100) for _ in range(30)]
df_two_annotators_continious = pd.DataFrame({'annotator_1': annotations_1, 'annotator_2': annotations_2})

In [ ]:
from scipy.stats import pearsonr
pearson_corr, pval = pearsonr(df_two_annotators_continious['annotator_1'], df_two_annotators_continious['annotator_2'])
print(f"Pearson Correlation: {pearson_corr}")
print(f"P-value: {pval}")


In [ ]:
# compute ICC using the pingouin library
import pingouin as pg
# first we need to convert the dataframe to long format. one column for the item id, one column for the annotator id, and one column for the score.
df_long = pd.melt(df_two_annotators_continious.reset_index(), id_vars=['index'], value_vars=['annotator_1', 'annotator_2'], var_name='annotator', value_name='score')
df_long.head()

In [ ]:
# compute ICC
icc = pg.intraclass_corr(data=df_long, targets='index', raters='annotator', ratings='score')
icc_31 = icc[icc['Type'] == 'ICC3']['ICC'].values[0]
print("ICC Results:")
print(icc_31)
# show the ICC(3,1) value

### Krippendorff's Alpha

The most robust agreement metric is the krippendorf's alpha, which can handle any number of annotators, missing data, and different types of data (nominal, ordinal, interval, ratio). A value of 1 indicates perfect agreement, while a value of 0 indicates no agreement beyond chance.

In [ ]:
import krippendorff
# use the dataframe with three annotators from above. the library krippendorff requires the data to be in a specific format: a list of lists, where each inner list represents the annotations for a single item. so each row in the dataframe needs to be converted to a list which contains the annotations from all annotators.
data = df_three_annotators_categorical.values.T.tolist()
alpha = krippendorff.alpha(reliability_data=data, level_of_measurement='nominal')

In [ ]:
print("Krippendorff's Alpha:", alpha)

### Your Task:
Now it's your turn to calculate the agreement between you and your colleagues' annotations. You can combine your annotations into one dataframe and use the correct method from above to compute the agreement metric. Make sure to choose the appropriate metric based on the number of annotators and the type of data you have annotated.